# Active learning and CDU ablation study

This notebook reconstructs the curves and tables that compare Random, Sum, Avg, DUA, and CDU from `run_model/results/active_learning_results.csv`. The protocol starts with 252 labeled images and adds 20 images per round for 60 rounds. Validation and test sets remain fixed. In every round, the detector is reinitialized from the same pretrained YOLOv9e weights; the comparison therefore measures the effect of sample selection without confounding it with cumulative fine-tuning.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

START = Path.cwd().resolve()
CANDIDATE_ROOTS = (START, *START.parents, START / 'repo')
ROOT = next((path for path in CANDIDATE_ROOTS
             if (path / 'run.py').is_file()
             and (path / 'active_learning_benchmark').is_dir()), None)
if ROOT is None:
    raise RuntimeError('The repository root could not be located.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from active_learning_benchmark.paper_plots import active_learning_figures

artifacts = active_learning_figures(ROOT / 'run_model')
FIGURES = artifacts['figures']


## Efficiency over the annotation budget

The acquired-image view keeps the same budget for every strategy. The object-based view shows that batches with the same number of images can require different numbers of annotated boxes. The figures show one point every six rounds, while the AUC integrates all available observations against the common empirical ceiling of 0.4269996823. CDU reached 0.8117, above DUA (0.8082), Avg (0.7791), and Sum (0.7687).


In [ ]:
display(Image(filename=str(FIGURES / 'graph_iterations_vs_map50.png')))
display(Image(filename=str(FIGURES / 'graph_instances_vs_map50.png')))
display(artifacts['active_learning_auc'].style.format({'normalized_auc': '{:.4f}'}))


## Test-set performance

The table below reproduces the final comparison. Avg obtained the highest terminal mAP@50, while CDU achieved the highest precision (0.497), the highest AUC, and the best `crack` mAP@50. With the same budget of 1,200 acquired images, CDU accumulated 2,441 objects, approximately 6.04% fewer than DUA (2,598), indicating that the trajectory gain did not depend only on more densely annotated images.


In [ ]:
results_table = artifacts['tab_results']
display(results_table.style.format({
    'precision': '{:.3f}', 'mAP50': '{:.3f}', 'mAP50_crack': '{:.3f}',
    'mAP50_patch': '{:.3f}', 'mAP50_pothole': '{:.3f}', 'normalized_auc': '{:.3f}',
}, na_rep='--'))


## Class-wise behavior during acquisition

Bars represent accumulated objects by class, while lines show class-wise mAP@50 at the points displayed every six rounds. Among these displayed points, DUA achieved its highest `crack` result with 1,080 acquired images, whereas CDU reached approximately 0.053 with 840 images. The complete records between displayed points remain in the CSV and contribute to AUC. The subsequent decrease at the summarized points is interpreted as possible saturation of new and diverse information in the pool, rather than simply insufficient detector capacity.


In [ ]:
display(Image(filename=str(FIGURES / 'bar_DUA.png')))
display(Image(filename=str(FIGURES / 'bar_CDU.png')))


## Nine CDU variants

The ablation isolates group-number optimization, proportional redistribution by Sainte-Laguë, candidate-set expansion, and diversity-aware refinement. The final configuration combines multiplication factor 4, Sainte-Laguë, and α=0.55. It does not maximize terminal mAP@50, but it achieves the highest trajectory AUC (0.812) and a `crack` AUC of 0.4364, consistent with using the budget efficiently throughout the cycle. For the `crack` curve, each method contributes one observation per round and AUC is normalized by the highest mAP@50 among these same nine variants; no trajectory outside the ablation changes the denominator.


In [ ]:
variant_table = artifacts['tab_results_model'].drop(
    columns=['available_iterations'], errors='ignore'
)
metrics = ['precision', 'mAP50', 'mAP50_crack', 'mAP50_patch', 'mAP50_pothole', 'normalized_auc']

def highlight_top_two(column):
    values = column.dropna().sort_values(ascending=False).drop_duplicates()
    styles = pd.Series('', index=column.index)
    if len(values):
        styles[column == values.iloc[0]] = 'font-weight: bold'
    if len(values) > 1:
        styles[column == values.iloc[1]] = 'text-decoration: underline'
    return styles

display(variant_table.style.format({metric: '{:.3f}' for metric in metrics})
        .apply(highlight_top_two, subset=metrics))


In [ ]:
display(Image(filename=str(FIGURES / 'AUC_crack.png')))
visible_crack_auc = artifacts['auc_crack'].drop(
    columns=['available_iterations'], errors='ignore'
)
display(visible_crack_auc.style.format({
    'normalized_auc_crack': '{:.4f}', 'normalizer': '{:.6f}',
}))
